# Terra Lens — Pipeline B: Fine-Tune-Only, Serial (pretrained SwinIR + edge-guided monkeypatch + serial FRN)

**Team TechTonic.** This is the pragmatic hackathon path: reuse a pretrained SwinIR
checkpoint almost entirely as-is, patch in edge-guided attention via a lightweight
monkeypatch, fine-tune only a handful of parameters on SEN2VENµS B2/B3/B4/B8, then
bolt on a small **serial** frequency-refinement head (FRN) trained independently.
No joint dual-branch training, no risk of destabilizing the pretrained backbone.

**Steps:**
1. Load pretrained SwinIR (classical SR, x2) — official weights, RGB, 3-channel.
2. Band-inflate `conv_first`/`conv_last` from 3 -> 4 channels (B2,B3,B4,B8), mapping
   pretrained R/G/B filters directly and initializing NIR from the Red filter.
3. Monkeypatch edge-guided modulation into every RSTB block (adds one `_edge_alpha`
   scalar per block — cheap, shape-preserving, doesn't touch pretrained weights).
4. Fine-tune **only** `conv_first`, `conv_last`, the `_edge_alpha` scalars (optionally
   the last 1-2 RSTB blocks) at a small LR, few epochs.
5. Freeze the fine-tuned SwinIR. Train a small **serial** FRN head on top of its
   *output* (not the raw LR input): block-DCT -> 2-3 conv layers -> inverse DCT,
   added back with a single learned scalar. Frequency loss + light SAM only.

Run top-to-bottom on a Kaggle GPU kernel with internet **on** (Settings -> Internet)
so it can fetch the pretrained SwinIR checkpoint.


In [ ]:
# ============================================================
# 0. Environment setup (Kaggle GPU kernel: Settings -> Accelerator -> GPU T4x2 / P100)
# ============================================================
import os, sys, math, glob, random, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# rasterio handles Sentinel-2 / VENuS GeoTIFF band reads
try:
    import rasterio
except ImportError:
    os.system("pip install -q rasterio")
    import rasterio


## Dataset setup on Kaggle — SEN2VENµS

SEN2VENµS pairs Sentinel-2 L1C reflectance (resampled to 5 m) with VENµS
reference imagery (native ~5 m, radiometrically/geometrically corrected) across
29 sites. Terra Lens targets exactly **B2 (Blue), B3 (Green), B4 (Red), B8 (NIR)**.

**Steps to get it onto this kernel:**

1. On kaggle.com, open *Datasets → New Dataset* (or search first — SEN2VENµS is
   already mirrored by several public Kaggle datasets; search `"sen2venus"`).
   If nothing usable turns up, create your own mirror:
   - Download the site archives from the official host (IEEE DataPort /
     Zenodo mirror of SEN2VENµS, `https://zenodo.org/record/...` — check the
     paper's data-availability section for the current DOI).
   - Upload the per-site `.zip`/`.nc`/`.tif` archives as a private Kaggle
     Dataset (`kaggle datasets create -p <local_dir>` via the Kaggle CLI, or
     the web uploader — Kaggle datasets support up to ~100 GB per dataset,
     which comfortably covers all 29 sites).
2. In this notebook: **Add Data → your SEN2VENµS dataset** (right-hand panel).
   It will mount read-only at `/kaggle/input/<dataset-slug>/`.
3. Point `DATA_ROOT` below at that mount. The loader below expects, per site,
   a folder containing the four bands for both sensors:
   `S2_<site>_<date>_B{02,03,04,08}.tif` (5 m Sentinel-2) and
   `VENUS_<site>_<date>_B{...}.tif` (VENµS reference) — adjust `GLOB_PATTERNS`
   to match whatever the mirror you find actually ships; SEN2VENµS is
   distributed as per-patch `.nc`/`.pt`/`.npy` bundles depending on the
   mirror, so this cell is written defensively (tries GeoTIFF first, falls
   back to `.npy`/`.pt` stacks of shape `[4,H,W]` in band order B2,B3,B4,B8).
4. **Split by site, not by patch.** Patches from the same site/date are
   spatially correlated (same field boundaries, same cloud/atmosphere), so a
   patch-level split leaks information across train/val/test and inflates
   your reported PSNR/SSIM. The code below hashes site names into
   train/val/test buckets deterministically.


In [ ]:
# ============================================================
# 1. Dataset: SEN2VENµS, split by SITE (not by patch) to avoid leakage
# ============================================================
DATA_ROOT = "/kaggle/input/sen2venus"   # <-- point this at your mounted dataset
BANDS = ["B02", "B03", "B04", "B08"]     # Blue, Green, Red, NIR  (Terra Lens target)

def discover_sites(root):
    """Each subfolder under root is treated as one site/date acquisition."""
    sites = sorted([d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)])
    if not sites:
        # flat layout fallback: group files by the <site> token in the filename
        files = glob.glob(os.path.join(root, "*"))
        sites = sorted(set("_".join(os.path.basename(f).split("_")[:2]) for f in files))
    return sites

def site_level_split(sites, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = random.Random(seed)
    sites = sites[:]
    rng.shuffle(sites)
    n = len(sites)
    n_test = max(1, int(n * test_frac))
    n_val = max(1, int(n * val_frac))
    test_sites = sites[:n_test]
    val_sites = sites[n_test:n_test + n_val]
    train_sites = sites[n_test + n_val:]
    return train_sites, val_sites, test_sites

def _load_band_stack(path_no_ext):
    """Try GeoTIFF (per-band files), then .npy, then .pt — whatever the mirror ships."""
    tif_paths = [f"{path_no_ext}_{b}.tif" for b in BANDS]
    if all(os.path.exists(p) for p in tif_paths):
        arrs = []
        for p in tif_paths:
            with rasterio.open(p) as src:
                arrs.append(src.read(1).astype(np.float32))
        return np.stack(arrs, axis=0)
    if os.path.exists(path_no_ext + ".npy"):
        return np.load(path_no_ext + ".npy").astype(np.float32)
    if os.path.exists(path_no_ext + ".pt"):
        return torch.load(path_no_ext + ".pt").numpy().astype(np.float32)
    raise FileNotFoundError(f"No band stack found for {path_no_ext}")

class Sen2VenusPairs(Dataset):
    """
    Returns (LR, HR) 4-band (B2,B3,B4,B8) reflectance pairs, normalized to [0,1]
    via a fixed reflectance scale (Sentinel-2 L1C digital numbers /10000).
    LR = Sentinel-2 side (5 m), HR = VENµS side (native, ~2.3x finer).
    """
    def __init__(self, site_dirs, patch_size=96, scale=2, augment=True, refl_scale=10000.0):
        self.pairs = []
        for sd in site_dirs:
            lr_candidates = sorted(glob.glob(os.path.join(sd, "*S2*")))
            hr_candidates = sorted(glob.glob(os.path.join(sd, "*VENUS*")))
            # naive pairing by index; adjust to real filename convention once known
            for lr, hr in zip(lr_candidates, hr_candidates):
                self.pairs.append((lr, hr))
        self.patch_size = patch_size
        self.scale = scale
        self.augment = augment
        self.refl_scale = refl_scale

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        lr = _load_band_stack(os.path.splitext(lr_path)[0]) / self.refl_scale
        hr = _load_band_stack(os.path.splitext(hr_path)[0]) / self.refl_scale

        ps = self.patch_size
        h, w = lr.shape[-2:]
        y = random.randint(0, max(0, h - ps))
        x = random.randint(0, max(0, w - ps))
        lr_p = lr[:, y:y + ps, x:x + ps]
        hr_p = hr[:, y * self.scale:(y + ps) * self.scale, x * self.scale:(x + ps) * self.scale]

        if self.augment:
            if random.random() < 0.5:
                lr_p = np.ascontiguousarray(lr_p[:, :, ::-1]); hr_p = np.ascontiguousarray(hr_p[:, :, ::-1])
            if random.random() < 0.5:
                lr_p = np.ascontiguousarray(lr_p[:, ::-1, :]); hr_p = np.ascontiguousarray(hr_p[:, ::-1, :])
            k = random.randint(0, 3)
            lr_p = np.ascontiguousarray(np.rot90(lr_p, k, axes=(1, 2)))
            hr_p = np.ascontiguousarray(np.rot90(hr_p, k, axes=(1, 2)))

        return torch.from_numpy(lr_p.copy()).float(), torch.from_numpy(hr_p.copy()).float()

sites_all = discover_sites(DATA_ROOT)
print(f"Discovered {len(sites_all)} candidate sites under {DATA_ROOT}")
train_sites, val_sites, test_sites = site_level_split(sites_all)
print(f"train sites: {len(train_sites)} | val sites: {len(val_sites)} | test sites: {len(test_sites)}")


## 2. Degradation model (for the Wald-protocol evaluation leg — same as Notebook A)

In [ ]:
# ============================================================
# 2. Degradation model — Wald's protocol operators B (blur), S (subsample), R (sensor response)
#    Used to synthesize LR/reference pairs when no real 5m -> ~2.5m ground truth exists.
# ============================================================
def gaussian_kernel2d(kernel_size=9, sigma=1.6, device="cpu"):
    ax = torch.arange(kernel_size, device=device) - (kernel_size - 1) / 2.0
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    k = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return (k / k.sum()).float()

class WaldDegradation(nn.Module):
    """
    B: sensor-MTF-matched Gaussian blur (per-band, depthwise conv, fixed).
    S: nearest/area subsampling by `scale`.
    R: sensor spectral-response perturbation (mild per-band gain jitter),
       applied only during training for robustness, off at eval time.
    """
    def __init__(self, n_bands=4, scale=2, kernel_size=9, sigma=1.6, jitter=0.02):
        super().__init__()
        k = gaussian_kernel2d(kernel_size, sigma)
        self.register_buffer("kernel", k.view(1, 1, kernel_size, kernel_size).repeat(n_bands, 1, 1, 1))
        self.scale = scale
        self.n_bands = n_bands
        self.jitter = jitter
        self.pad = kernel_size // 2

    def forward(self, x, training_jitter=False):
        # B: blur
        xb = F.conv2d(F.pad(x, [self.pad] * 4, mode="reflect"), self.kernel, groups=self.n_bands)
        # S: subsample
        xs = xb[:, :, ::self.scale, ::self.scale]
        # R: sensor-response gain jitter (train-time only, simulates cross-sensor mismatch)
        if training_jitter:
            gain = 1.0 + (torch.rand(1, self.n_bands, 1, 1, device=x.device) - 0.5) * 2 * self.jitter
            xs = xs * gain
        return xs

def bicubic_upsample(x, scale):
    return F.interpolate(x, scale_factor=scale, mode="bicubic", align_corners=False)


## 3. Load pretrained SwinIR (official architecture + weights)

We clone the official `JingyunLiang/SwinIR` repo so the model class matches the
released checkpoint's `state_dict` keys exactly (reimplementing SwinIR from
scratch risks silent key-mismatches that `load_state_dict(strict=False)` would
hide). If your Kaggle kernel has no internet, instead **Add Data** a mirrored
"SwinIR pretrained weights" Kaggle dataset and point `SWINIR_CKPT` at it, and
`pip install` a vendored copy of `network_swinir.py` as a Kaggle Utility Script.


In [ ]:
# ---- Fetch official SwinIR repo + pretrained classical-SR x2 checkpoint ----
REPO_DIR = "/kaggle/working/SwinIR"
if not os.path.isdir(REPO_DIR):
    os.system(f"git clone -q https://github.com/JingyunLiang/SwinIR.git {REPO_DIR}")
sys.path.insert(0, REPO_DIR)
from models.network_swinir import SwinIR

SWINIR_CKPT_URL = ("https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/"
                    "001_classicalSR_DIV2K_s48w8_SwinIR-M_x2.pth")
SWINIR_CKPT = "/kaggle/working/swinir_x2.pth"
if not os.path.exists(SWINIR_CKPT):
    os.system(f"wget -q -O {SWINIR_CKPT} {SWINIR_CKPT_URL}")

# Config must match the checkpoint (classical SR x2, 6 RSTB depth, window 8, embed_dim 180)
SCALE = 2
swinir = SwinIR(
    upscale=SCALE, in_chans=3, img_size=48, window_size=8, img_range=1.0,
    depths=[6, 6, 6, 6, 6, 6], embed_dim=180, num_heads=[6, 6, 6, 6, 6, 6],
    mlp_ratio=2, upsampler="pixelshuffle", resi_connection="1conv",
).to(DEVICE)

state = torch.load(SWINIR_CKPT, map_location=DEVICE)
state = state.get("params", state)
missing, unexpected = swinir.load_state_dict(state, strict=False)
print(f"Loaded pretrained SwinIR. missing={len(missing)} unexpected={len(unexpected)}")


## 4. Band-informed inflation: conv_first / conv_last from 3 -> 4 channels (B2,B3,B4,B8)

In [ ]:
# ============================================================
# 4. Band-informed inflation: adapt an RGB-pretrained conv stem to B2/B3/B4/B8
#    B4(Red)->R, B3(Green)->G, B2(Blue)->B, B8(NIR) initialized from the Red filter
#    (NIR edge structure correlates with red-channel structure more than G/B).
# ============================================================
@torch.no_grad()
def inflate_rgb_conv_to_b2b3b4b8(conv_rgb: nn.Conv2d) -> nn.Conv2d:
    """
    conv_rgb: nn.Conv2d with in_channels == 3, weight layout [out,3,kh,kw] assumed RGB order.
    Returns a new nn.Conv2d with in_channels == 4, band order [B2,B3,B4,B8] = [Blue,Green,Red,NIR].
    """
    assert conv_rgb.in_channels == 3
    w = conv_rgb.weight.data  # [out, 3, kh, kw] in R,G,B order
    out_c, _, kh, kw = w.shape
    new_w = torch.zeros(out_c, 4, kh, kw, dtype=w.dtype, device=w.device)
    R, G, B = w[:, 0], w[:, 1], w[:, 2]
    new_w[:, 0] = B         # B2 Blue  <- pretrained Blue filter
    new_w[:, 1] = G         # B3 Green <- pretrained Green filter
    new_w[:, 2] = R         # B4 Red   <- pretrained Red filter
    new_w[:, 3] = R.clone() # B8 NIR   <- pretrained Red filter (edge-structure proxy, not RGB average)

    new_conv = nn.Conv2d(4, out_c, (kh, kw), stride=conv_rgb.stride, padding=conv_rgb.padding,
                          bias=(conv_rgb.bias is not None))
    new_conv.weight.data.copy_(new_w)
    if conv_rgb.bias is not None:
        new_conv.bias.data.copy_(conv_rgb.bias.data)
    return new_conv


In [ ]:
@torch.no_grad()
def inflate_output_conv_to_4band(conv_rgb: nn.Conv2d) -> nn.Conv2d:
    """Same band mapping as the input stem, applied to the *output* projection (conv_last)."""
    assert conv_rgb.out_channels == 3
    w = conv_rgb.weight.data  # [3, in, kh, kw]
    in_c, kh, kw = w.shape[1:]
    new_w = torch.zeros(4, in_c, kh, kw, dtype=w.dtype, device=w.device)
    R, G, B = w[0], w[1], w[2]
    new_w[0], new_w[1], new_w[2], new_w[3] = B, G, R, R.clone()
    new_conv = nn.Conv2d(in_c, 4, (kh, kw), stride=conv_rgb.stride, padding=conv_rgb.padding,
                          bias=(conv_rgb.bias is not None))
    new_conv.weight.data.copy_(new_w)
    if conv_rgb.bias is not None:
        b = conv_rgb.bias.data
        new_bias = torch.stack([b[2], b[1], b[0], b[0].clone()])
        new_conv.bias.data.copy_(new_bias)
    return new_conv

# conv_first: Conv2d(3, embed_dim, 3,1,1);  conv_last: Conv2d(num_feat_out, 3, 3,1,1) for pixelshuffle head
swinir.conv_first = inflate_rgb_conv_to_b2b3b4b8(swinir.conv_first).to(DEVICE)
swinir.conv_last = inflate_output_conv_to_4band(swinir.conv_last).to(DEVICE)
print("conv_first:", swinir.conv_first, "\nconv_last:", swinir.conv_last)


## 5. Monkeypatch: edge-guided modulation into every RSTB block

Adds one learnable `_edge_alpha` scalar per RSTB that gates how much the block's
*residual update* gets amplified near edges (Sobel-derived, sigma-gated map of the
input). Shape-preserving, doesn't touch any pretrained weight — purely additive —
so it's safe to drop into a frozen pretrained backbone.


In [ ]:
import types

class SobelEdge(nn.Module):
    def __init__(self, n_bands):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        ky = kx.t()
        self.register_buffer("kx", kx.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.register_buffer("ky", ky.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.n_bands = n_bands

    def forward(self, x):
        gx = F.conv2d(F.pad(x, [1, 1, 1, 1], mode="reflect"), self.kx, groups=self.n_bands)
        gy = F.conv2d(F.pad(x, [1, 1, 1, 1], mode="reflect"), self.ky, groups=self.n_bands)
        e = torch.sqrt(gx ** 2 + gy ** 2 + 1e-6).mean(dim=1, keepdim=True)
        return torch.sigmoid(e)

class EdgeGuidedRSTBWrapper(nn.Module):
    """Wraps a pretrained RSTB. out = rstb(x) + alpha * edge_gate * (rstb(x) - x)."""
    def __init__(self, rstb):
        super().__init__()
        self.rstb = rstb
        self._edge_alpha = nn.Parameter(torch.tensor(0.1))

    def forward(self, x, x_size, edge_gate=None):
        out = self.rstb(x, x_size)
        if edge_gate is not None:
            out = out + self._edge_alpha * edge_gate * (out - x)
        return out

def install_edge_guidance(model, n_bands=4):
    device = next(model.parameters()).device
    model._edge_sobel = SobelEdge(n_bands).to(device)
    model._edge_cache = None

    # wrap every RSTB in-place
    for i, layer in enumerate(model.layers):
        model.layers[i] = EdgeGuidedRSTBWrapper(layer)

    def patched_forward_features(self, x):
        x_size = (x.shape[2], x.shape[3])
        if self._edge_cache is not None:
            eg = F.interpolate(self._edge_cache, size=x_size, mode="bilinear", align_corners=False)
            eg_flat = eg.flatten(2).transpose(1, 2)         # (B, H*W, 1) — broadcasts over channels
        else:
            eg_flat = None
        xt = self.patch_embed(x)
        if self.ape:
            xt = xt + self.absolute_pos_embed
        xt = self.pos_drop(xt)
        for layer in self.layers:
            xt = layer(xt, x_size, edge_gate=eg_flat)
        xt = self.norm(xt)
        xt = self.patch_unembed(xt, x_size)
        return xt
    model.forward_features = types.MethodType(patched_forward_features, model)

    orig_forward = model.forward
    def patched_forward(self, x):
        self._edge_cache = self._edge_sobel(x)
        return orig_forward(x)
    model.forward = types.MethodType(patched_forward, model)
    return model

swinir = install_edge_guidance(swinir, n_bands=4)
print("Edge guidance installed. RSTB wrappers:", sum(isinstance(m, EdgeGuidedRSTBWrapper) for m in swinir.layers))


## 6. Selective fine-tuning

Freeze everything except `conv_first`, `conv_last`, every `_edge_alpha` scalar,
and (optionally) the last 1-2 RSTB blocks. The pretrained backbone already does
the heavy lifting; this converges fast at a tiny LR.


In [ ]:
FINE_TUNE_LAST_N_RSTB = 2   # set to 0 to fine-tune only conv_first/conv_last/_edge_alpha
n_rstb = len(swinir.layers)

for name, p in swinir.named_parameters():
    p.requires_grad = False

for p in swinir.conv_first.parameters():
    p.requires_grad = True
for p in swinir.conv_last.parameters():
    p.requires_grad = True
for name, p in swinir.named_parameters():
    if name.endswith("_edge_alpha"):
        p.requires_grad = True

if FINE_TUNE_LAST_N_RSTB > 0:
    for i in range(n_rstb - FINE_TUNE_LAST_N_RSTB, n_rstb):
        for p in swinir.layers[i].parameters():
            p.requires_grad = True

trainable_params = [p for p in swinir.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in swinir.parameters())
print(f"Fine-tuning {n_trainable/1e3:.1f}K / {n_total/1e6:.2f}M params "
      f"({100*n_trainable/n_total:.2f}%)")


In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))

charbonnier = CharbonnierLoss()

BATCH_SIZE = 8
PATCH_SIZE = 96
EPOCHS_FT = 12
LR_FT = 1e-5

train_ds = Sen2VenusPairs(train_sites, patch_size=PATCH_SIZE, scale=SCALE, augment=True)
val_ds = Sen2VenusPairs(val_sites, patch_size=PATCH_SIZE, scale=SCALE, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

opt = torch.optim.AdamW(trainable_params, lr=LR_FT, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FT)


In [ ]:
# ============================================================
# 5. Metrics — Wald protocol leg (needs simulated GT) + full-res leg (no GT needed)
# ============================================================
def psnr(sr, hr, max_val=1.0):
    mse = F.mse_loss(sr, hr)
    return 10 * torch.log10((max_val ** 2) / (mse + 1e-10))

def rmse(sr, hr):
    return torch.sqrt(F.mse_loss(sr, hr) + 1e-12)

def sam(sr, hr, eps=1e-8):
    # Spectral Angle Mapper, averaged over pixels, in degrees
    dot = (sr * hr).sum(dim=1)
    n1 = sr.norm(dim=1); n2 = hr.norm(dim=1)
    cos = (dot / (n1 * n2 + eps)).clamp(-1 + 1e-7, 1 - 1e-7)
    ang = torch.acos(cos)
    return ang.mean() * 180.0 / math.pi

def ergas(sr, hr, scale, eps=1e-8):
    # Erreur Relative Globale Adimensionnelle de Synthese
    n_bands = sr.shape[1]
    total = 0.0
    for b in range(n_bands):
        rmse_b = torch.sqrt(F.mse_loss(sr[:, b], hr[:, b]) + eps)
        mean_b = hr[:, b].mean() + eps
        total += (rmse_b / mean_b) ** 2
    return 100.0 / scale * torch.sqrt(total / n_bands)

def ndvi(x, red_idx=2, nir_idx=3, eps=1e-6):
    red, nir = x[:, red_idx], x[:, nir_idx]
    return (nir - red) / (nir + red + eps)

def ndvi_consistency(sr, hr, red_idx=2, nir_idx=3):
    """Domain-specific check: how well does SR preserve NDVI vs the reference (no hallucinated vegetation edges)."""
    return F.l1_loss(ndvi(sr, red_idx, nir_idx), ndvi(hr, red_idx, nir_idx))

@torch.no_grad()
def wald_protocol_eval(model, loader, degradation, scale, device=DEVICE):
    model.eval()
    stats = {"psnr": [], "rmse": [], "sam": [], "ergas": [], "ndvi": []}
    for lr_native, hr_native in loader:
        # Wald: degrade one scale further so the *native* LR/HR pair itself becomes verifiable ground truth
        hr = hr_native.to(device)
        lr_sim = degradation(hr, training_jitter=False).to(device)
        sr, _ = model(lr_sim)
        stats["psnr"].append(psnr(sr, hr).item())
        stats["rmse"].append(rmse(sr, hr).item())
        stats["sam"].append(sam(sr, hr).item())
        stats["ergas"].append(ergas(sr, hr, scale).item())
        stats["ndvi"].append(ndvi_consistency(sr, hr).item())
    return {k: float(np.mean(v)) for k, v in stats.items()}

@torch.no_grad()
def full_res_eval(model, loader, degradation, device=DEVICE):
    """
    No ground truth at full resolution: check consistency residuals.
    e_h: SR downsampled back should match the real high-res-side input (if available).
    e_m: SR degraded through B,S,R should match the original LR input (self-consistency, Wald invariance).
    """
    model.eval()
    eh_list, em_list = [], []
    for lr_native, hr_native in loader:
        lr_native = lr_native.to(device)
        sr, _ = model(lr_native)
        # e_m: degrade SR back down, compare to original LR
        sr_degraded = degradation(sr, training_jitter=False)
        em = F.l1_loss(sr_degraded, lr_native).item()
        em_list.append(em)
        # e_h: (only if a genuine independent high-res reference exists for this patch)
        if hr_native is not None and hr_native.numel() > 0:
            hr_native = hr_native.to(device)
            eh = F.l1_loss(F.interpolate(sr, size=hr_native.shape[-2:], mode="bicubic", align_corners=False),
                            hr_native).item()
            eh_list.append(eh)
    out = {"e_m": float(np.mean(em_list))}
    if eh_list:
        out["e_h"] = float(np.mean(eh_list))
    return out


In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
best_psnr = -1

for epoch in range(1, EPOCHS_FT + 1):
    swinir.train()
    running = 0.0
    t0 = time.time()
    for lr_b, hr_b in train_loader:
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        sr_b = swinir(lr_b)
        sr_b = F.interpolate(sr_b, size=hr_b.shape[-2:], mode="bicubic", align_corners=False) \
               if sr_b.shape[-2:] != hr_b.shape[-2:] else sr_b
        loss = charbonnier(sr_b, hr_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        opt.step()
        running += loss.item()
    sched.step()

    if epoch % 2 == 0 or epoch == EPOCHS_FT:
        swinir.eval()
        with torch.no_grad():
            vlr, vhr = next(iter(val_loader))
            vlr, vhr = vlr.to(DEVICE), vhr.to(DEVICE)
            vsr = swinir(vlr)
            vsr = F.interpolate(vsr, size=vhr.shape[-2:], mode="bicubic", align_corners=False) \
                  if vsr.shape[-2:] != vhr.shape[-2:] else vsr
            vpsnr = psnr(vsr, vhr).item()
        print(f"[FineTune][epoch {epoch:02d}/{EPOCHS_FT}] loss={running/len(train_loader):.4f} "
              f"| val PSNR={vpsnr:.2f} | {time.time()-t0:.1f}s")
        if vpsnr > best_psnr:
            best_psnr = vpsnr
            torch.save(swinir.state_dict(), os.path.join(CKPT_DIR, "swinir_finetuned_best.pt"))


## 7. Serial FRN refinement head — trained alone, SwinIR frozen

Feeds the fine-tuned SwinIR's **output** through block-DCT -> a small 2-3 layer
conv net -> inverse block-DCT, added back with a single learned scalar. No
AC-energy gate here (that's for fusing two co-equal branches; this is refining
an already-good output, which is a strictly simpler job). Trained with a
frequency-domain loss plus a light SAM term, SwinIR fully frozen.


In [ ]:
def _dct_basis(n=8):
    basis = np.zeros((n, n), dtype=np.float32)
    for k in range(n):
        for i in range(n):
            ck = math.sqrt(1 / n) if k == 0 else math.sqrt(2 / n)
            basis[k, i] = ck * math.cos(math.pi * (2 * i + 1) * k / (2 * n))
    return torch.from_numpy(basis)

class BlockDCT(nn.Module):
    def __init__(self, block=8):
        super().__init__()
        self.block = block
        D = _dct_basis(block)
        self.register_buffer("D", D)
        self.register_buffer("Dt", D.t())

    def forward(self, x, inverse=False):
        B, C, H, W = x.shape
        b = self.block
        pad_h = (b - H % b) % b
        pad_w = (b - W % b) % b
        xp = F.pad(x, [0, pad_w, 0, pad_h], mode="reflect")
        Hp, Wp = xp.shape[-2:]
        patches = xp.unfold(2, b, b).unfold(3, b, b)
        nH, nW = patches.shape[2], patches.shape[3]
        patches = patches.reshape(-1, b, b)
        D, Dt = (self.Dt, self.D) if inverse else (self.D, self.Dt)
        out = D @ patches @ Dt
        out = out.reshape(B, C, nH, nW, b, b).permute(0, 1, 2, 4, 3, 5).reshape(B, C, Hp, Wp)
        return out[:, :, :H, :W]

class SerialFRN(nn.Module):
    def __init__(self, n_bands=4, feat=24, block=8):
        super().__init__()
        self.dct = BlockDCT(block)
        self.body = nn.Sequential(
            nn.Conv2d(n_bands, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, feat, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(feat, n_bands, 3, 1, 1),
        )
        self.alpha = nn.Parameter(torch.tensor(0.1))

    def forward(self, sr):
        D = self.dct(sr, inverse=False)
        D_ref = self.body(D)
        refined = self.dct(D_ref, inverse=True)
        return sr + self.alpha * refined

def sam_loss(sr, hr, eps=1e-8):
    dot = (sr * hr).sum(dim=1)
    n1, n2 = sr.norm(dim=1), hr.norm(dim=1)
    cos = (dot / (n1 * n2 + eps)).clamp(-1 + 1e-7, 1 - 1e-7)
    return torch.acos(cos).mean()

frn = SerialFRN(n_bands=4).to(DEVICE)
for p in swinir.parameters():
    p.requires_grad = False
swinir.eval()

EPOCHS_FRN = 15
LR_FRN = 2e-4
opt_frn = torch.optim.AdamW(frn.parameters(), lr=LR_FRN)
sched_frn = torch.optim.lr_scheduler.CosineAnnealingLR(opt_frn, T_max=EPOCHS_FRN)
dct_for_loss = BlockDCT(8).to(DEVICE)


In [ ]:
best_frn_psnr = -1
for epoch in range(1, EPOCHS_FRN + 1):
    frn.train()
    running = 0.0
    for lr_b, hr_b in train_loader:
        lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
        with torch.no_grad():
            sr0 = swinir(lr_b)
            sr0 = F.interpolate(sr0, size=hr_b.shape[-2:], mode="bicubic", align_corners=False) \
                  if sr0.shape[-2:] != hr_b.shape[-2:] else sr0
        opt_frn.zero_grad(set_to_none=True)
        sr_ref = frn(sr0)
        l_freq = F.l1_loss(dct_for_loss(sr_ref), dct_for_loss(hr_b))
        l_sam = sam_loss(sr_ref, hr_b) * 0.05
        loss = l_freq + l_sam
        loss.backward()
        opt_frn.step()
        running += loss.item()
    sched_frn.step()

    if epoch % 3 == 0 or epoch == EPOCHS_FRN:
        frn.eval()
        with torch.no_grad():
            vlr, vhr = next(iter(val_loader))
            vlr, vhr = vlr.to(DEVICE), vhr.to(DEVICE)
            vsr0 = swinir(vlr)
            vsr0 = F.interpolate(vsr0, size=vhr.shape[-2:], mode="bicubic", align_corners=False) \
                   if vsr0.shape[-2:] != vhr.shape[-2:] else vsr0
            vsr_ref = frn(vsr0)
            vpsnr = psnr(vsr_ref, vhr).item()
        print(f"[FRN][epoch {epoch:02d}/{EPOCHS_FRN}] loss={running/len(train_loader):.4f} | val PSNR={vpsnr:.2f}")
        if vpsnr > best_frn_psnr:
            best_frn_psnr = vpsnr
            torch.save(frn.state_dict(), os.path.join(CKPT_DIR, "serial_frn_best.pt"))


## 8. Final evaluation — both legs (Wald protocol + full-res residuals)

In [ ]:
class PipelineB(nn.Module):
    def __init__(self, swinir_model, frn_model):
        super().__init__()
        self.swinir = swinir_model
        self.frn = frn_model
    def forward(self, lr):
        sr0 = self.swinir(lr)
        sr = self.frn(sr0)
        return sr, {"sr_pre_frn": sr0}

pipeline_b = PipelineB(swinir, frn).to(DEVICE).eval()

@torch.no_grad()
def wald_protocol_eval_b(model, loader, degradation, scale):
    stats = {"psnr": [], "rmse": [], "sam": [], "ergas": [], "ndvi": []}
    for lr_native, hr_native in loader:
        hr = hr_native.to(DEVICE)
        lr_sim = degradation(hr, training_jitter=False).to(DEVICE)
        sr, _ = model(lr_sim)
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)
        stats["psnr"].append(psnr(sr, hr).item())
        stats["rmse"].append(rmse(sr, hr).item())
        stats["sam"].append(sam(sr, hr).item())
        stats["ergas"].append(ergas(sr, hr, scale).item())
        stats["ndvi"].append(ndvi_consistency(sr, hr).item())
    return {k: float(np.mean(v)) for k, v in stats.items()}

@torch.no_grad()
def full_res_eval_b(model, loader, degradation):
    em_list = []
    for lr_native, _ in loader:
        lr_native = lr_native.to(DEVICE)
        sr, _ = model(lr_native)
        sr_degraded = degradation(sr, training_jitter=False)
        if sr_degraded.shape[-2:] != lr_native.shape[-2:]:
            sr_degraded = F.interpolate(sr_degraded, size=lr_native.shape[-2:], mode="bicubic", align_corners=False)
        em_list.append(F.l1_loss(sr_degraded, lr_native).item())
    return {"e_m": float(np.mean(em_list))}

test_ds = Sen2VenusPairs(test_sites, patch_size=PATCH_SIZE, scale=SCALE, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
wald_deg = WaldDegradation(n_bands=4, scale=SCALE).to(DEVICE)

wald_results = wald_protocol_eval_b(pipeline_b, test_loader, wald_deg, SCALE)
fullres_results = full_res_eval_b(pipeline_b, test_loader, wald_deg)

print("=== Wald protocol eval (simulated GT) ===")
for k, v in wald_results.items():
    print(f"  {k.upper():8s}: {v:.4f}")
print("\n=== Full-resolution eval (no GT, consistency residual) ===")
for k, v in fullres_results.items():
    print(f"  {k}: {v:.4f}")

with open("/kaggle/working/pipelineB_results.json", "w") as f:
    json.dump({"wald_protocol": wald_results, "full_res": fullres_results}, f, indent=2)


## 9. Quick visual sanity check

In [ ]:
import matplotlib.pyplot as plt

lr_b, hr_b = next(iter(test_loader))
lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)
with torch.no_grad():
    sr_b, aux = pipeline_b(lr_b)
    if sr_b.shape[-2:] != hr_b.shape[-2:]:
        sr_b = F.interpolate(sr_b, size=hr_b.shape[-2:], mode="bicubic", align_corners=False)

def to_rgb(t):
    rgb = t[[2, 1, 0]].clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return rgb

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_rgb(F.interpolate(lr_b[0:1], size=hr_b.shape[-2:], mode="bicubic")[0])); axes[0].set_title("Bicubic LR")
axes[1].imshow(to_rgb(sr_b[0])); axes[1].set_title("Pipeline B SR (fine-tuned + serial FRN)")
axes[2].imshow(to_rgb(hr_b[0])); axes[2].set_title("Reference (VENµS)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/pipelineB_sample.png", dpi=150)
plt.show()
